### Protocol: API Financial Scanner & Advanced Visualizer
**Objective:** Implement a batch-processing script to extract and dynamically visualize live corporate financial data.
**Data Source:** Yahoo Finance API (`yfinance`).
**Technical Specifications:**
* Batch processing for multiple tickers.
* `pandas` DataFrame matrix generation.
* `seaborn` and `matplotlib` for dynamic data rendering (conditional color mapping based on threshold analysis).

In [1]:
!pip install yfinance pandas matplotlib seaborn

### Data Ingestion & Algorithmic Scoring
* Fetches JSON payload for multiple tickers.
* Evaluates KPIs against predefined financial thresholds to generate a Health Score (0-4).

In [2]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, Any

def calculate_health_score(data: Dict[str, Any]) -> int:
    score = 0
    cr = data.get("current_ratio")
    if isinstance(cr, (int, float)) and cr >= 1.5: score += 1

    dte = data.get("debt_to_equity")
    if isinstance(dte, (int, float)) and dte <= 100.0: score += 1

    pm = data.get("profit_margin")
    if isinstance(pm, (int, float)) and pm >= 0.10: score += 1

    roe = data.get("return_on_equity")
    if isinstance(roe, (int, float)) and roe >= 0.10: score += 1

    return score

def fetch_corporate_data(ticker_symbol: str) -> Dict[str, Any]:
    try:
        company = yf.Ticker(ticker_symbol)
        data = company.info

        if not data or len(data) <= 1:
            return {"Ticker": ticker_symbol, "Error": "Invalid Ticker"}

        parsed_data = {
            "Ticker": ticker_symbol,
            "Name": data.get("shortName", "N/A"),
            "current_ratio": data.get("currentRatio", "N/A"),
            "debt_to_equity": data.get("debtToEquity", "N/A"),
            "profit_margin": data.get("profitMargins", "N/A"),
            "return_on_equity": data.get("returnOnEquity", "N/A")
        }

        parsed_data["Health_Score"] = calculate_health_score(parsed_data)
        return parsed_data

    except Exception as e:
        return {"Ticker": ticker_symbol, "Error": str(e)}

print("System: API, Scoring, and Visualization modules loaded.")

System: API, Scoring, and Visualization modules loaded.


### Execution, Matrix Generation & Visualization
* Compiles dictionaries into a sorted `pandas` DataFrame.
* Renders a professional 1x3 `seaborn` dashboard.
* Enhances analytics with conditional coloring (Green = Safe/Target Met, Red = High Risk/Target Missed).

In [ ]:
def format_percentage(value) -> str:
    if value == "N/A" or value is None: return "N/A"
    return f"{(value * 100):.1f}%"

def main():
    print("="*80)
    print(" YFINANCE API: QUANTITATIVE DASHBOARD")
    print("="*80)

    raw_input = input("Enter Stock Tickers separated by commas (e.g., AAPL, MSFT, TSLA, RHM.DE): ")
    tickers = [t.strip().upper() for t in raw_input.split(",") if t.strip()]

    if not tickers:
        print("Error: No valid tickers provided.")
        return

    results = []
    for ticker in tickers:
        print(f"Fetching data for {ticker}...")
        report = fetch_corporate_data(ticker)

        if "Error" not in report:
            raw_pm = report["profit_margin"] * 100 if isinstance(report["profit_margin"], (int, float)) else 0
            raw_dte = report["debt_to_equity"] if isinstance(report["debt_to_equity"], (int, float)) else 0

            formatted_report = {
                "Ticker": report["Ticker"],
                "Company": report["Name"],
                "Liquidität 3": report["current_ratio"] if report["current_ratio"] != "N/A" else "N/A",
                "Verschuldung (%)": report["debt_to_equity"] if report["debt_to_equity"] != "N/A" else "N/A",
                "Umsatzrendite": format_percentage(report["profit_margin"]),
                "ROE": format_percentage(report["return_on_equity"]),
                "Health Score (/4)": report["Health_Score"],
                "_Raw_PM": raw_pm,
                "_Raw_DTE": raw_dte
            }
            results.append(formatted_report)
        else:
            print(f"  -> Skipped {ticker}: {report['Error']}")

    if not results:
        return

    # DataFrame Generation
    df = pd.DataFrame(results).sort_values(by="Health Score (/4)", ascending=False).reset_index(drop=True)
    display_df = df.drop(columns=["_Raw_PM", "_Raw_DTE"])

    print("\n" + "="*80)
    print(" COMPARATIVE FINANCIAL MATRIX")
    print("="*80)
    print(display_df.to_string())
    print("="*80)

    # ---------------------------------------------------------
    # ADVANCED SEABORN DASHBOARD
    # ---------------------------------------------------------
    print("\nSystem: Rendering conditional analytical dashboard...")

    # Set professional visual theme
    sns.set_theme(style="whitegrid")
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle('Quantitative Financial Analysis - Peer Comparison', fontsize=16, fontweight='bold', y=1.05)

    # --- Chart 1: Health Score (Uniform Dark Blue) ---
    sns.barplot(x='Ticker', y='Health Score (/4)', data=df, ax=axes[0], color='#2c3e50')
    axes[0].set_title('Fundamental Health Score', fontweight='bold')
    axes[0].set_ylabel('Score (0-4)')
    axes[0].set_ylim(0, 4.5)

    # --- Chart 2: Profit Margin (Conditional: Green >= 10%, Red < 10%) ---
    pm_colors = ['#27ae60' if val >= 10 else '#e74c3c' for val in df['_Raw_PM']]
    sns.barplot(x='Ticker', y='_Raw_PM', data=df, ax=axes[1], palette=pm_colors)
    axes[1].axhline(y=10, color='#34495e', linestyle='--', linewidth=1.5, label='Target (10%)')
    axes[1].set_title('Umsatzrendite (Profit Margin)', fontweight='bold')
    axes[1].set_ylabel('Margin (%)')
    axes[1].legend()

    # --- Chart 3: Debt-to-Equity (Conditional: Green <= 100%, Red > 100%) ---
    dte_colors = ['#27ae60' if val <= 100 else '#e74c3c' for val in df['_Raw_DTE']]
    sns.barplot(x='Ticker', y='_Raw_DTE', data=df, ax=axes[2], palette=dte_colors)
    axes[2].axhline(y=100, color='#34495e', linestyle='--', linewidth=1.5, label='Safe Limit (100%)')
    axes[2].set_title('Verschuldungsgrad (Debt-to-Equity)', fontweight='bold')
    axes[2].set_ylabel('Debt / Equity (%)')
    axes[2].legend()

    # Apply data labels to all charts
    for ax, column in zip(axes, ['Health Score (/4)', '_Raw_PM', '_Raw_DTE']):
        for i, v in enumerate(df[column]):
            label = f"{v:.1f}%" if column != 'Health Score (/4)' else str(v)
            # Position label slightly above the bar dynamically
            offset = max(df[column]) * 0.03 if max(df[column]) > 0 else 1
            ax.text(i, v + offset, label, ha='center', va='bottom', fontweight='bold', color='black')

    # Cleanup borders
    sns.despine()
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    main()

 YFINANCE API: QUANTITATIVE DASHBOARD
